# EDA

No W&B. Run bootstrap first.

## 0) Bootstrap (Kaggle / Colab)

Run once per session: clone or update this repo, `cd` into it, install deps.

**Enable Internet** in the Kaggle notebook settings, or the clone and pip install fail.

`REPO_BRANCH` must point at the branch that holds the preprocessing code. Once
`feat/preprocessing` is merged into `main` you can set it back to `"main"`.

In [ ]:
import os, sys, subprocess
from pathlib import Path

REPO_URL = "https://github.com/tuan8p/VN-Traffic-Sign-Classification.git"
REPO_DIR = "VN-Traffic-Sign-Classification"
REPO_BRANCH = "feat/preprocessing"   # <- the preprocessing code lives here, not on main

base = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path.cwd()
repo_path = base / REPO_DIR


def sh(cmd: str, check: bool = True) -> int:
    print(">>", cmd)
    return subprocess.run(cmd, shell=True, check=check).returncode


if (Path.cwd() / ".git").exists() and (Path.cwd() / "configs" / "shared.yaml").exists():
    repo_path = Path.cwd()
    print("using the repo this notebook already sits in:", repo_path)
elif not (repo_path / ".git").exists():
    sh(f'git clone --branch {REPO_BRANCH} {REPO_URL} "{repo_path}"')

# Always land on REPO_BRANCH: a previous run may have cloned the default branch,
# which is what produces "cannot import name find_dataset_root".
git = f'git -C "{repo_path}"'
sh(f"{git} fetch origin {REPO_BRANCH}")
sh(f"{git} checkout -B {REPO_BRANCH} origin/{REPO_BRANCH}")

os.chdir(repo_path)
if str(repo_path) not in sys.path:
    sys.path.insert(0, str(repo_path))
print("cwd    :", Path.cwd())
sh(f"{git} log --oneline -1")

req = repo_path / "requirements-kaggle.txt"
sh(f'pip install -q -r "{req if req.exists() else repo_path / "requirements.txt"}"')
sh("pip install -q -e .")

# Fail loudly here rather than three cells later.
import importlib
for mod in ("vn_tsc.data.discover", "vn_tsc.data.crops", "vn_tsc.data.grouping",
            "vn_tsc.data.splitting", "vn_tsc.data.augment", "vn_tsc.data.classes"):
    importlib.reload(importlib.import_module(mod)) if mod in sys.modules else importlib.import_module(mod)
from vn_tsc.data.discover import find_dataset_root  # noqa: F401
print("bootstrap OK — preprocessing modules import cleanly")

## 1) Generate the figures

Reads `data/processed` (run `01_preprocess_offline.ipynb` first, or attach the
published dataset) and writes PNGs plus `eda_summary.json` to `analysis/figures`.

In [ ]:
from pathlib import Path
import json, pandas as pd
from analysis.eda import main as run_eda

PROCESSED = next(p for p in ["data/processed", "/kaggle/input/vnts-processed"]
                 if Path(p).exists())
FIGURES = "analysis/figures"
run_eda(["--processed", PROCESSED, "--out", FIGURES])

## 2) Headline numbers

In [ ]:
s = json.loads(Path(FIGURES, "eda_summary.json").read_text(encoding="utf-8"))
print(f"real crops         : {s['n_crops_real']} from {s['n_source_images']} source images")
print(f"augmented crops    : {s['n_crops_augmented']}")
print(f"classes            : {s['n_classes']}")
print(f"imbalance          : {s['imbalance_ratio']}:1")
print(f"rare classes (<30) : {s['n_rare_classes_below_30']} -> {s['rare_classes_below_30']}")
print(f"mirror pairs       : {s['mirror_pairs']}   <- hflip is unsafe")
print(f"leakage, dataset   : {s['leakage']['provided_by_authors']['n_affected_images']} images")
print(f"leakage, ours      : {s['leakage']['ours']['n_affected_images']} images")

per_class = pd.read_csv(Path(FIGURES, "per_class_counts.csv"))
cols = ["code", "name_vi", "train", "val", "test", "train_aug", "total_real"]
print("\nrarest classes")
display(per_class.nsmallest(8, "total_real")[cols])
print("most common classes")
display(per_class.nlargest(5, "total_real")[cols])

## 3) The figures

1. class distribution — the imbalance
2. augmentation effect — train before/after
3. box sizes — how small the signs really are
4. position heatmap — where signs sit in the frame
5. leakage — duplicate clusters, dataset split vs ours
6. one example per class — manual label check
7. mirror pairs — why horizontal flip is banned
8. confusable groups — classes differing only in fine detail
9. brightness and sharpness — the lighting and blur spread

In [ ]:
from IPython.display import Image, display as show
for png in sorted(Path(FIGURES).glob("*.png")):
    print("=" * 78)
    print(png.name)
    show(Image(filename=str(png)))